## Configuration and Imports

In [ ]:
from datasets import load_dataset
from tqdm import tqdm

# --- CONFIGURATION ---
DATASET_PATH = "HuggingFaceH4/ultrachat_200k"
DATASET_SPLIT = "train_sft"
SAMPLE_SIZE = 100000

CODE_KEYWORDS = {
    "def ",
    "class ",
    "import ",
    "return ",
    "print(",
    "function",
    "python",
    "javascript",
    "html",
    "css",
    "c++",
    "java",
    "sql",
    "compile",
    "debug",
    "variable",
    "array",
    "syntax",
    "algorithm",
    "```",
}
MATH_KEYWORDS = {
    "calculate",
    "equation",
    "theorem",
    "integral",
    "derivative",
    "algebra",
    "geometry",
    "math",
    "\\frac",
    "\\sqrt",
    "sum of",
}
MED_KEYWORDS = {
    "symptom",
    "diagnosis",
    "treatment",
    "patient",
    "syndrome",
    "mg ",
    "dosage",
    "clinical",
    "doctor",
    "disease",
    "infection",
}


## EDA

In [ ]:
def analyze_ultrachat_contamination():
    print(f"Streaming {SAMPLE_SIZE} prompts from UltraChat...")
    ds = load_dataset(DATASET_PATH, split=DATASET_SPLIT, streaming=True)

    stats = {
        "total_analyzed": 0,
        "contains_code": 0,
        "contains_math": 0,
        "contains_med": 0,
        "clean_general": 0,
    }

    for row in tqdm(ds, total=SAMPLE_SIZE, desc="Analyzing"):
        if stats["total_analyzed"] >= SAMPLE_SIZE:
            break

        messages = row.get("messages", [])
        if not messages or messages[0]["role"] != "user":
            continue

        prompt = messages[0]["content"].lower()
        stats["total_analyzed"] += 1

        # Check contamination
        has_code = any(kw in prompt for kw in CODE_KEYWORDS)
        has_math = any(kw in prompt for kw in MATH_KEYWORDS)
        has_med = any(kw in prompt for kw in MED_KEYWORDS)

        if has_code:
            stats["contains_code"] += 1
        if has_math:
            stats["contains_math"] += 1
        if has_med:
            stats["contains_med"] += 1

        if not (has_code or has_math or has_med):
            stats["clean_general"] += 1

    # Print Report
    print("\n" + "=" * 50)
    print("ULTRACHAT CONTAMINATION REPORT")
    print("=" * 50)
    total = stats["total_analyzed"]
    print(f"Total Prompts Analyzed: {total:,}")
    print("-" * 50)

    code_pct = (stats["contains_code"] / total) * 100
    math_pct = (stats["contains_math"] / total) * 100
    med_pct = (stats["contains_med"] / total) * 100
    clean_pct = (stats["clean_general"] / total) * 100

    print(
        f"Contains CODE keywords:    {stats['contains_code']:>5,} ({code_pct:>5.1f}%)"
    )
    print(
        f"Contains MATH keywords:    {stats['contains_math']:>5,} ({math_pct:>5.1f}%)"
    )
    print(f"Contains MEDICAL keywords: {stats['contains_med']:>5,} ({med_pct:>5.1f}%)")
    print("-" * 50)
    print(
        f"CLEAN (No target keywords):{stats['clean_general']:>5,} ({clean_pct:>5.1f}%)"
    )
    print("=" * 50)


## Execution

In [ ]:
if __name__ == "__main__":
    analyze_ultrachat_contamination()
